In [1]:
# ==============================
# CONFIG
# ==============================

DATASET = "nyc_tlc_yellow"

SILVER_PATH = f"../lakehouse/silver/{DATASET}"
GOLD_PATH = f"../lakehouse/gold/{DATASET}"

YEAR = 2024
MONTH = 1  # 1-12

GOLD_TABLE_KPI_DAILY = "kpi_daily_overview"

print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("YEAR/MONTH:", YEAR, MONTH)

SILVER_PATH: ../lakehouse/silver/nyc_tlc_yellow
GOLD_PATH: ../lakehouse/gold/nyc_tlc_yellow
YEAR/MONTH: 2024 1


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYC-TLC-Lakehouse")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .getOrCreate()
)

spark

26/02/25 09:52:19 WARN Utils: Your hostname, willian-A320M-S2H resolves to a loopback address: 127.0.1.1; using 192.168.0.110 instead (on interface enp6s0)
26/02/25 09:52:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/willian/PhaifferTech/nyc-tlc-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/willian/.ivy2/cache
The jars for the packages stored in: /home/willian/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1833b6bc-2c7c-4b79-bd4a-03041f12fb94;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 285ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |  

In [3]:
from pyspark.sql import functions as F

df_silver = spark.read.format("delta").load(SILVER_PATH)

silver_rows = df_silver.count()
print("silver_rows:", silver_rows)

bounds = df_silver.select(
    F.min("pickup_date").alias("min_pickup_date"),
    F.max("pickup_date").alias("max_pickup_date"),
).collect()[0]

print("silver_min_pickup_date:", bounds["min_pickup_date"])
print("silver_max_pickup_date:", bounds["max_pickup_date"])

# Small preview for sanity
df_silver.select("pickup_date").distinct().orderBy("pickup_date").show(5, truncate=False)

26/02/25 09:52:34 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/02/25 09:52:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


silver_rows: 2839382


silver_min_pickup_date: 2024-01-01
silver_max_pickup_date: 2024-01-31


+-----------+
|pickup_date|
+-----------+
|2024-01-01 |
|2024-01-02 |
|2024-01-03 |
|2024-01-04 |
|2024-01-05 |
+-----------+
only showing top 5 rows



In [4]:
from pyspark.sql.functions import col, unix_timestamp, coalesce, lit

# Handle schema drift across years safely.
# If a revenue column does not exist, create it deterministically as zero.
def safe_col(df, name, default_value=0.0):
    if name in df.columns:
        return coalesce(col(name).cast("double"), lit(float(default_value)))
    return lit(float(default_value))

df_enriched = (
    df_silver
    .withColumn(
        "trip_duration_sec",
        (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))).cast("long")
    )
    .withColumn("trip_duration_min", (col("trip_duration_sec") / lit(60.0)).cast("double"))
    # Normalized revenue components (nulls -> 0)
    .withColumn("fare_amount_z", safe_col(df_silver, "fare_amount"))
    .withColumn("tip_amount_z", safe_col(df_silver, "tip_amount"))
    .withColumn("tolls_amount_z", safe_col(df_silver, "tolls_amount"))
    .withColumn("extra_z", safe_col(df_silver, "extra"))
    .withColumn("mta_tax_z", safe_col(df_silver, "mta_tax"))
    .withColumn("improvement_surcharge_z", safe_col(df_silver, "improvement_surcharge"))
    .withColumn("congestion_surcharge_z", safe_col(df_silver, "congestion_surcharge"))
    .withColumn("airport_fee_z", safe_col(df_silver, "airport_fee"))
    .withColumn(
        "total_revenue",
        col("fare_amount_z")
        + col("tip_amount_z")
        + col("tolls_amount_z")
        + col("extra_z")
        + col("mta_tax_z")
        + col("improvement_surcharge_z")
        + col("congestion_surcharge_z")
        + col("airport_fee_z")
    )
)

df_enriched.select("pickup_date", "trip_duration_min", "total_revenue").show(5, truncate=False)

+-----------+------------------+-------------+
|pickup_date|trip_duration_min |total_revenue|
+-----------+------------------+-------------+
|2024-01-28 |3.2666666666666666|11.58        |
|2024-01-28 |3.5166666666666666|10.8         |
|2024-01-28 |22.55             |35.11        |
|2024-01-28 |8.266666666666667 |17.5         |
|2024-01-28 |12.95             |15.14        |
+-----------+------------------+-------------+
only showing top 5 rows



In [5]:
from pyspark.sql.functions import avg, sum, count, round as sround

kpi_daily = (
    df_enriched
    .groupBy("pickup_date")
    .agg(
        count(lit(1)).alias("trips"),
        sum("fare_amount_z").alias("total_fare"),
        sum("tip_amount_z").alias("total_tip"),
        sum("total_revenue").alias("total_revenue"),
        avg("trip_distance").alias("avg_trip_distance"),
        avg("trip_duration_min").alias("avg_trip_duration_min"),
        avg("passenger_count").alias("avg_passenger_count"),
    )
    .withColumn("avg_trip_distance", sround(col("avg_trip_distance"), 3))
    .withColumn("avg_trip_duration_min", sround(col("avg_trip_duration_min"), 3))
    .withColumn("avg_passenger_count", sround(col("avg_passenger_count"), 3))
)

kpi_daily.orderBy("pickup_date").show(10, truncate=False)

+-----------+-----+------------------+------------------+------------------+-----------------+---------------------+-------------------+
|pickup_date|trips|total_fare        |total_tip         |total_revenue     |avg_trip_distance|avg_trip_duration_min|avg_passenger_count|
+-----------+-----+------------------+------------------+------------------+-----------------+---------------------+-------------------+
|2024-01-01 |74830|1662841.4800000603|256689.6099999985 |2324594.019999933 |4.665            |16.689               |1.568              |
|2024-01-02 |72341|1548093.8100000108|259731.07999999783|2255838.3599998797|4.218            |17.063               |1.432              |
|2024-01-03 |79188|1590700.8200000047|271635.90999999957|2341511.009999927 |3.956            |16.677               |1.395              |
|2024-01-04 |99066|1860543.840000016 |332253.1699999981 |2787599.5399999293|3.367            |16.175               |1.371              |
|2024-01-05 |99019|1799324.4899999816|325

In [6]:
import os
import shutil

KPI_DAILY_PATH = f"{GOLD_PATH}/kpi_daily_overview"

if os.path.exists(KPI_DAILY_PATH):
    shutil.rmtree(KPI_DAILY_PATH)

(
    kpi_daily
    .write
    .format("delta")
    .mode("overwrite")
    .save(KPI_DAILY_PATH)
)

print("Saved:", KPI_DAILY_PATH)

Saved: ../lakehouse/gold/nyc_tlc_yellow/kpi_daily_overview


In [7]:
df_gold = spark.read.format("delta").load(KPI_DAILY_PATH)
print("gold_days:", df_gold.count())
df_gold.orderBy("pickup_date").show(31, truncate=False)

# Validate day count for a monthly window (portfolio-friendly guardrail)
assert 28 <= df_gold.count() <= 31, f"Unexpected number of days for a month window: {df_gold.count()}"

gold_days: 31


+-----------+------+------------------+------------------+------------------+-----------------+---------------------+-------------------+
|pickup_date|trips |total_fare        |total_tip         |total_revenue     |avg_trip_distance|avg_trip_duration_min|avg_passenger_count|
+-----------+------+------------------+------------------+------------------+-----------------+---------------------+-------------------+
|2024-01-01 |74830 |1662841.4800000603|256689.6099999985 |2324594.019999933 |4.665            |16.689               |1.568              |
|2024-01-02 |72341 |1548093.8100000108|259731.07999999783|2255838.3599998797|4.218            |17.063               |1.432              |
|2024-01-03 |79188 |1590700.8200000047|271635.90999999957|2341511.009999927 |3.956            |16.677               |1.395              |
|2024-01-04 |99066 |1860543.840000016 |332253.1699999981 |2787599.5399999293|3.367            |16.175               |1.371              |
|2024-01-05 |99019 |1799324.489999

In [8]:
from pathlib import Path

from pipelines.gold_marts.run_gold_enforced import run_gold_enforced

repo_root = Path.cwd().resolve()
if not (repo_root / "contracts").exists():
    repo_root = repo_root.parent

run_gold_enforced(
    spark=spark,
    repo_root=str(repo_root),
    contract_dataset="gold.fct_trips_daily",
    input_table="silver.trips_clean",
    output_table="gold.fct_trips_daily",
    quarantine_table="quality.quarantine_records",
    max_invalid_ratio=0.001,
    year=YEAR,
    month=MONTH,
)

print("Gold table ready: gold.fct_trips_daily")
print("Gold row count:", spark.table("gold.fct_trips_daily").count())


Gold table written: gold.fct_trips_daily


Gold row count: 31
